CAPA BRONCE

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS platform_dev;

CREATE SCHEMA IF NOT EXISTS platform_dev.bronce_byma;

CREATE TABLE IF NOT EXISTS platform_dev.bronce_byma.control_cargas (
    proceso STRING,
    ultima_fecha_procesada DATE,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.columnMapping.mode' = 'name',
  'delta.enableDeletionVectors' = 'true',
  'delta.logRetentionDuration' = '30 days'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.bronce_byma.calidad_duplicados (
    id_transaccion STRING,
    count_dup BIGINT,
    fecha_deteccion TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.autoCompact' = 'auto',
    'delta.columnMapping.mode' = 'name',
    'delta.enableDeletionVectors' = 'true',
    'delta.logRetentionDuration' = '30 days'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.bronce_byma.operaciones_raw (
    fecha STRING,
    tipoTran STRING,
    id_cliente STRING,
    descripcion_titulo STRING,
    moneda STRING,
    simbolo_titulo STRING,
    cantidad STRING,
    precio STRING,
    id_transaccion STRING,
    origen STRING,
    fecha_particion DATE,
    _ingested_at TIMESTAMP
)
    USING DELTA
    PARTITIONED BY (fecha_particion)
    TBLPROPERTIES (
        'delta.autoOptimize.autoCompact' = 'auto',
        'delta.columnMapping.mode' = 'name',
        'delta.enableDeletionVectors' = 'true',
        'delta.logRetentionDuration' = '30 days'
)

CAPA SILVER

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS platform_dev.silver_byma;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.silver_byma.operaciones_replica (
    fecha STRING,
    tipoTran STRING,
    id_cliente STRING,
    descripcion_titulo STRING,
    moneda STRING,
    simbolo_titulo STRING,
    cantidad decimal(18,4),
    precio decimal(18,4),
    id_transaccion STRING,
    origen STRING,
    fecha_particion DATE,
    _ingested_at TIMESTAMP,
    fecha_utc TIMESTAMP,
    fecha_local TIMESTAMP,
    _processed_at TIMESTAMP
)
    USING DELTA
    PARTITIONED BY (fecha_particion)
    TBLPROPERTIES (
        'delta.autoOptimize.autoCompact' = 'auto',
        'delta.columnMapping.mode' = 'name',
        'delta.enableDeletionVectors' = 'true',
        'delta.logRetentionDuration' = '30 days'
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.silver_byma.cotizaciones_historicas (
    simbolo STRING,
    fecha DATE,
    precio_cierre DECIMAL(18,4),
    precio_apertura DECIMAL(18,4),
    fuente_api STRING,
    _processed_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.autoCompact' = 'auto',
    'delta.columnMapping.mode' = 'name',
    'delta.enableDeletionVectors' = 'true',
    'delta.logRetentionDuration' = '30 days'
);

CAPA GOLD

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS platform_dev.gold_byma;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.gold_byma.dim_instrumento (
    instrumento_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    simbolo STRING,
    descripcion STRING,
    tipo_instrumento STRING,
    moneda_base STRING,
    ticker_api STRING,
    _created_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.gold_byma.dim_cliente (
    cliente_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    id_cliente STRING,
    segmento_actividad STRING,
    _created_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.gold_byma.dim_fecha (
    fecha_sk INT,
    fecha DATE,
    anio INT,
    mes INT,
    trimestre INT,
    dia_semana STRING,
    es_fin_de_semana BOOLEAN,
    es_feriado_bursatil BOOLEAN
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.gold_byma.dim_canal (
    canal_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    nombre_canal STRING,
    tipo_canal STRING,
    _created_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS platform_dev.gold_byma.fact_operaciones (
    id_transaccion STRING,
    instrumento_sk BIGINT,
    cliente_sk BIGINT,
    fecha_sk INT,
    canal_sk BIGINT,
    cantidad DECIMAL(18,4),
    precio_operado DECIMAL(18,4),
    monto_total DECIMAL(28,8),
    tipo_operacion STRING,
    precio_mercado DECIMAL(18,4),
    desvio_pct DECIMAL(18,4),
    flag_compro_caro BOOLEAN,
    flag_vendio_barato BOOLEAN,
    _processed_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (fecha_sk);